In [18]:
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

db = duckdb.connect(
    "../../Data/retail.db",
    read_only=True
)
db.sql("SET search_path TO mart;")
db.sql("""SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'mart'""")

┌─────────────────┐
│   table_name    │
│     varchar     │
├─────────────────┤
│ customerdim     │
│ productdim      │
│ retail_enriched │
└─────────────────┘

NameError: name 'os' is not defined

# Revenue Overview

Uppdrag att ta fram data på försäljningen

* Hur mycket säljer vi för?
* Hur ser intäkterna ut över tid?
* Finns det säsongsvariation i intäkterna?


Notering:
Inga större filter tillagda för att visa totala omsättning.
December 2011 exkluderas från månadsvyerna då endast 9 dagars data finns tillgängligt.

In [19]:
# Totala intäkter inkluderar alla positiva transaktioner. 
df_dataset_revenue = db.sql("SELECT ROUND(SUM(total_price)) as Total_Dataset_Revenue FROM retail_enriched WHERE total_price > 0 ").df()


In [20]:
# intäkter uppdelat på år
df_revenue = db.sql("""SELECT
       YEAR(invoicedate) as Year,
       ROUND(SUM(total_price), 2) as revenue
       FROM retail_enriched
       WHERE total_price > 0 
       AND is_product = 1
       GROUP by 1
       ORDER BY 1
       """).df()


In [21]:
# Graf med intäkter per år
px.bar(
    df_revenue,
    x='Year',
    y='revenue',
    title='Revenue per Year'
)

Datan sträcker sig mellan 2010-12-01 och 2011-12-09 och gör djupare säsongsanalyser omöjliga. Fokus ligger främst på intäktsutveckling, månads och veckovisa trender under datasettets period.

Totala intäkter under perioden uppgår till £10642111.
£775286.18 från december 2010 och £9480687.56 från 2011. 

In [22]:
# Månatliga intäkter
df_rev_month = db.sql("""
       SELECT
       DATE_TRUNC('Month', invoicedate) as month,
       SUM(total_price) as revenue
       FROM retail_enriched
       WHERE total_price > 0 
       AND DATE_TRUNC('month', invoicedate) < '2011-12-01'
       GROUP BY 1
       ORDER BY 1
       """).df()


In [23]:
# Graf över intäkter per månad
rev_month_fig = px.line(
    df_rev_month,
    x='month',
    y='revenue',
    title='Revenue per Month'    
)
# Skriver till HTML för användning på github pages
rev_month_fig.write_html(
    "../../Analysis/HTML/monthly_revenue.html",
    include_plotlyjs='cdn'
    )
rev_month_fig.show()


Högsäsong inom retail tenderar att infalla runt november och december inför högtider. Detta speglas i försäljningen som i de 3 första kvartalen trendar svagt uppåt för att sedan accelerera in i november.
November blir den klart starkaste månaden i datasettet där försäljning mot föregående månad ökade med hela 30%. 

In [24]:
#Intäkter per vecka
df_rev_week = db.sql("""
       SELECT
       DATE_TRUNC('week', invoicedate) as week_start,
       SUM(total_price) as revenue
       FROM retail_enriched
       WHERE total_price > 0 
       GROUP BY 1
       ORDER BY 1
       """).df()

In [25]:
# Graf över veckovisa intäkter
fig_rev_week = px.line(
    df_rev_week,
    x='week_start',
    y='revenue',
    title='Revenue per Week'
)
fig_rev_week.write_html("../../Analysis/HTML/weekly_revenue.html",
    include_plotlyjs='cdn')
fig_rev_week.show()

In [26]:
# Skillnad i veckovisa intäkter mot föregående vecka
df_rev_week_growth = db.sql("""
                            WITH a AS (
                            SELECT
                            DATE_TRUNC('week', invoicedate) as week,
                            SUM(total_price) as revenue
                            FROM retail_enriched
                            WHERE total_price > 0
                            GROUP BY 1
                            )

                            SELECT
                            week,
                            revenue,
                            LAG(revenue) OVER(ORDER BY week) as lag_revenue,
                            (ROUND((revenue - LAG(revenue) OVER(ORDER BY week)) / LAG(revenue) OVER(ORDER BY week), 3) * 100) as weekly_growth
                            FROM a
                            ORDER BY week 
                            
                            """).df()

In [27]:
# Graf på förändring i intäkter vecka till vecka
fig_wow_rev_growth = px.line(
    df_rev_week_growth,
    x='week',
    y='weekly_growth',
    title='Revenue week over week growth'
)
fig_wow_rev_growth.write_html("../../Analysis/HTML/wow_revenue_growth.html",
    include_plotlyjs='cdn')
fig_wow_rev_growth.show()

Veckovisa intäkter tillåter oss att jämföra den överlappande period i datasettets start och slut. Mellan godtyckligt jämförbara perioder har försäljningen ökat med 64% på årsbasis. 

2010-12-01 till 2010-12-13 -- 184669.470 + 323398.700
2011-11-28 till 2011-12-09 -- 329108.220 + 503767.750

värt att notera: Trots stabil intäktstillväxt så går det att urskilja viss betydfande fluktuation i intäkter månad till månad och vecka till vecka. 


In [28]:
#intäkter CAGR över hela datasettet
first_week_rev = df_rev_week_growth['revenue'].iloc[0]
last_week_rev = df_rev_week_growth['revenue'].iloc[-1]
n_week = len(df_rev_week_growth) - 1

weekly_cagr = (last_week_rev / first_week_rev) ** (1/n_week) - 1


Den sammansatta veckovisa tillväxten under datasettets period uppgår till 1,95%. Detta motsvarar närmare en fördubbling av veckovisa intäkter under perioden. Tillsammans med de 64% i tillväxt som jämförelsen mellan de två perioderna i 2010 och 2011 hintade om tyder det på att 2011 var ett mycket bra år. 

In [29]:
# Orders per månad
df_order_month = db.sql("""
                        SELECT
                        DATE_TRUNC('Month', invoicedate) as month,
                        COUNT(DISTINCT invoiceno) as orders
                        FROM retail_enriched
                        WHERE is_credit_invoice = 0
                        AND DATE_TRUNC('month', invoicedate) < '2011-12-01'
                        GROUP BY 1
                        ORDER BY 1
                        """).df()

In [30]:
# Graf över order per månad
fig_order_month = px.line(df_order_month, x='month', y='orders', title='Orders per Month')
fig_order_month.write_html("../../Analysis/HTML/monthly_orders.html",
    include_plotlyjs='cdn')
fig_order_month.show()

In [31]:
# Genomsnittlig intäkt per order per månad
df_rev_order_m = db.sql("""
                      SELECT
                      DATE_TRUNC('month', invoicedate) as month,
                      SUM(total_price)/COUNT(DISTINCT invoiceno) as rev_per_order
                      FROM retail_enriched
                      WHERE total_price > 0
                      AND DATE_TRUNC('month', invoicedate) < '2011-12-01'
                      GROUP BY 1
                      ORDER BY 1
                      """).df()

In [32]:
# Genomsnittlig intäkt per order över hela datasettet
df_avg_rev_order = db.sql("""
                      SELECT
                      SUM(total_price)/COUNT(DISTINCT invoiceno) as rev_per_order
                      FROM retail_enriched
                      WHERE total_price > 0
                      """).df()

In [33]:
# Graf över intäkt per order per månad
fig_rev_order_m = px.line(df_rev_order_m, x='month', y='rev_per_order', title='Revenue per Order by Month')
fig_rev_order_m.write_html("../../Analysis/HTML/monthly_revenue_order.html",
    include_plotlyjs='cdn')
fig_rev_order_m.show()

Orderläggningen har i stort sett följt intäkter under året och tyder på att tillväxten drivits av fler köp totalt. De marginella avvikelserna mellan orders och intäkter  indikerar viss fluktuation av ordervärde.

Januari 2011 sticker däremot ut med ett extra högt ordervärde på £635, vilket är 19% över periodens genomsnitt. Trots en av de svagare månaderna sett till antal orders (1216) så var januari tack vare dessa större orders en bra månad. 
Det är möjligt att detta fenomen i januari är en effekt av nytt år. Ett temporärt skifte i kundbeteende. Kunder kommer tillbaka från ledigheter, lager ska fyllas på, nya produkter ska testas. 

# Sammanfattning
* Intäkter visar tydlig trend uppåt under 2011
* Tillväxt är främst driven av fler orders
* intäkter per order fortsätter vara relativt stabil med undantag för januari
* Stark säsongseffekt som peakar i november
* Viss kortsiktig fluktuation i intäkter på veckobasis

    Revisit

Efter att ha bekantat mig mer med datan i andra analyser så finns det delar som kan förbättras. 
Samlar några här.

    Potentiella förbättringsåtgärder (uppd nedan)
* Initiala analys inkluderade alla positiva transaktioner. Även sådana som troligtvis inte hörde till försäljning.
    - En stor del av dessa inkluderas i "staging.top_anomalies".


In [34]:
# Justerad data för veckovisa intäkter. Filtrerar bort poster som rimligen inte hör till försäljning.
df_rev_week_adj = db.sql("""
       SELECT
       DATE_TRUNC('week', invoicedate) as week_start,
       SUM(total_price) as revenue
       FROM retail_enriched
       WHERE total_price > 0 
       AND customerid IS NOT NULL
       AND description NOT ILIKE '%POSTAGE%'
       AND description NOT ILIKE '%AMAZON%'
       AND description NOT ILIKE '%Manual%'
       GROUP BY 1
       ORDER BY 1
       """).df() 

In [35]:
# Plottar graf med justerad data
fig_rev_week_adj = px.line(
    df_rev_week_adj,
    x='week_start',
    y='revenue',
    title='Revenue per Week'
)
fig_rev_week_adj.write_html("../../Analysis/HTML/weekly_revenue_adj.html",
    include_plotlyjs='cdn')
fig_rev_week_adj.show()